# 04 — Materials: Decision Tree vs Random Forest

Train on `data/materials` and `data/labels` CSVs; report accuracy, precision, recall, F1, confusion matrix; cost roll-up for a toy selection.

In [ ]:
from pathlib import Path
import sys

ROOT = Path().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from blueprint_estimator.features import build_pair_features, load_materials_table
from blueprint_estimator.train_trees import train_decision_tree, train_random_forest
from blueprint_estimator.cost import rollup_estimate

materials = load_materials_table(ROOT / "data" / "materials" / "materials_sample.csv")
labels = pd.read_csv(ROOT / "data" / "labels" / "labels_sample.csv")
X, y, joined = build_pair_features(materials, labels)
cat = ["material_type_req", "material_type"]

dt = train_decision_tree(X, y, cat)
rf = train_random_forest(X, y, cat)
print("--- Decision Tree ---")
print("F1:", dt["f1"], "best_params:", dt["best_params"])
print(dt["confusion_matrix"])
print("--- Random Forest ---")
print("F1:", rf["f1"], "best_params:", rf["best_params"])
print(rf["confusion_matrix"])

model = rf["model"]
proba = model.predict_proba(X)[:, 1]
ranked = joined.assign(proba=proba)
top = ranked.sort_values("proba", ascending=False).groupby("pair_id", as_index=False).head(1)
print("Top pick per pair_id (by proba):\n", top[["pair_id", "material_id", "match", "proba"]])

selections = [(row.material_id, 120.0, "lf") for row in materials.head(3).itertuples()]
total, detail = rollup_estimate(materials, selections)
print("Toy cost rollup total USD:", round(total, 2))
detail